In [1]:
%pip install -q dotenv llama_stack_client==0.4.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [3]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")

client = LlamaStackClient(
    base_url=base_url
)

In [4]:
# List available shields
shields = client.shields.list()
print("Available shields:")
for s in shields:
    print(f"  - {s.identifier} (provider: {s.provider_id})")

INFO:httpx:HTTP Request: GET http://llamastack-distribution-service:8321/v1/shields "HTTP/1.1 200 OK"


Available shields:
  - content_safety (provider: llama-guard)
  - hap (provider: trustyai_fms)
  - pii (provider: trustyai_fms)
  - prompt_injection (provider: trustyai_fms)


In [5]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = "You are a helpful assistant."
SHIELDS = ["pii", "hap"]

In [6]:
def handle_message(msg):
    """Send a message using responses API with guardrails via extra_body (streaming)."""
    try:
        stream = client.responses.create(
            model=MODEL,
            input=msg,
            instructions=INSTRUCTIONS,
            stream=True,
            extra_body={"guardrails": SHIELDS},
        )

        for event in stream:
            event_type = getattr(event, "type", None)

            if event_type == "response.output_text.delta":
                print(event.delta, end="", flush=True)

            elif event_type == "response.refusal.delta":
                print(event.delta, end="", flush=True)

            elif event_type == "response.completed":
                response = event.response
                for item in response.output:
                    if item.type == "message" and hasattr(item, "content"):
                        for part in item.content:
                            if part.type == "refusal":
                                print(f"\n  SHIELD VIOLATION: Blocked by safety policy.")
                                print(f"    Message: {part.refusal}")
                                return

        print()  # newline after streaming

    except Exception as e:
        print(f"\n  Error: {e}")

In [7]:
msg = "Give me a sentence that contains the word: aloha"
handle_message(msg)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, the user wants a sentence with the word "aloha." Let me think about how to use it naturally. "Aloha" is a Hawaiian word meaning hello or goodbye. Maybe start with a greeting. "Aloha, friend, it's so good to see you!" That sounds friendly. Wait, should I add more context? Maybe mention the island vibe. "Aloha, friend, it's so good to see you! The warm breeze and gentle waves of the Hawaiian island make every moment feel special." That adds some atmosphere. Is there a better way to incorporate it? Maybe use it in a different context, like a message. "Sending you aloha and good vibes from across the ocean!" But the user might prefer a more descriptive sentence. Let me check if the first example is clear and natural. Yeah, that works. I'll go with that.
</think>

"Aloha, friend, it's so good to see you! The warm breeze and gentle waves of the Hawaiian island make every moment feel special."


In [8]:
msg = "You dotard, I really hate this"
handle_message(msg)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"



  SHIELD VIOLATION: Blocked by safety policy.
    Message: You are a helpful assistant. You dotard, I really hate this (flagged for: LABEL_1)
